# Etap 3, część 1 — błąd pojedynczego odsypu (test na siatce)

Udajemy odsyp bez masy w każdej pełnej sekundzie fragmentów bez zdarzeń: **błąd = wartość z okna 3 s przed − wartość z okna 3 s po** (po końcu czasu trwania T i przerwie d). Prawdziwa masa wynosi 0, więc wynik to sam błąd z dryfu i szumu wskazań. T = 10, 20, 30, 45, 60, 90 s, d = 0, 2, 3 s, trzy metody (9 wariantów jak w etapie 2; domyślnie mediana, d = 3 s). Fragmenty: co najmniej 60 s od prawdziwych zdarzeń (przyjętych przez reguły i przyjętych w przeglądzie odsypów), bez problemów z katalogu i luk w próbkach. Dzień 5–22, noc 22–5 (czas lokalny).

Przedziały 95 %: bootstrap po dobach (doba = jednostka losowania, dzień i noc osobno), B = 2000. Plan: `plans/dms-data-analysis/etap3-miary-bledu-plan.md` §3. Obliczenia: `uv run python -m dms_analysis.errors grid`.

In [ ]:
from IPython.display import Markdown, display

from dms_analysis import style
from dms_analysis.data.catalog import load_catalog
from dms_analysis.errors import browser, scaling, store, views

style.register()
results = views.load_all(load_catalog())
summary = store.load_summary("swietochlowice-1")["grid_stats"]
for warning in views.warnings(results):
    display(Markdown(f"<span style='color:#e34948'><b>⚠ {warning}</b></span>"))

## 1. Dane wejściowe i kontrola

Ile startów daje każdy okres, i porównanie z liczbami z dyskusji 15.11 (odchylenie std, okresy `0004` + `0005`, mediana, d = 3 s).

In [ ]:
views.inputs_table(results)

In [ ]:
views.control_table(results)

## 2. Miary błędu — okresy z czasem rzeczywistym razem

Wiersze: czas trwania T. Kolor = wartość (im ciemniej, tym większy błąd). Percentyle liczone z |błędu|; średni błąd ze znakiem (dodatni: wskazanie spadło w trakcie).

In [ ]:
views.measures_table(summary)

Odchylenie std i p95 w 9 wariantach dla T = 30 s (czy metoda i przerwa coś zmieniają).

In [ ]:
views.variants_table(summary)

In [ ]:
views.variants_table(summary, measure="p95")

## 3. Porównanie między T

Miary w kg rosną z T, więc nie da się ich porównać wprost. Dwa sposoby:

**Błąd względny** — błąd jako procent typowej masy odsypu o czasie trwania T (masa = mediana szybkości odsypu z etapu 2 × T). Odpowiada na pytanie, jak duży jest błąd względem tego, co ważymy. Założenie: stała szybkość odsypu.

In [ ]:
rate = scaling.dispense_rate(views.real_time(results))
scaling.relative_table(summary, rate)

**Jak błąd rośnie z T** — obie osie logarytmiczne, więc zależność błąd ∝ T^k jest prostą o nachyleniu k. Nachylenie 0: sam szum (błąd nie zależy od T), 0,5: dryf losowy (∝ √T), 1: stały dryf w jedną stronę (∝ T). Krzywa płaska na początku i rosnąca dalej = szum przy krótkich T, dryf przy długich. Nachylenie pośrednie może być też mieszanką szumu i stałego dryfu — wykres pokazuje, który składnik przeważa, nie dowodzi rodzaju dryfu. p95 leży na siatce działki (20 kg), więc jego krzywa idzie schodkami.

In [ ]:
scaling.scaling_figure(summary).show()

Nachylenie między kolejnymi T (log zmiany błędu / log zmiany T) i dla całego zakresu 10→90 s.

In [ ]:
scaling.slopes(summary)

## 4. Rozkład błędu

In [ ]:
views.histogram_figure(results).show()

## 5. Błąd wg godziny

Okresy z czasem rzeczywistym; fioletowe pasma = noc. Najechanie pokazuje liczbę startów w danej godzinie.

In [ ]:
views.by_hour_figure(results).show()

## 6. Okresy osobno

Okres z czasem szacowanym (±2 h) liczony, ale bez podziału wg godziny — tu tylko dla porównania (pora dnia może być przesunięta).

In [ ]:
views.periods_table(summary, results)

## 7. Pokrycie

Liczba startów w każdej godzinie każdej doby (T = 30 s). Pełna godzina bez zdarzeń ≈ 3600; jasne komórki = mało danych (zdarzenia, luki, problemy z katalogu).

In [ ]:
views.coverage_figure(results).show()

## 8. Najgorsze starty

20 największych |błędów| na okres × T × porę (mediana, d = 3 s), co najmniej 120 s od siebie (każdy to osobny epizod, nie kolejne sekundy tego samego), na tle wskazań z oknami.

In [ ]:
browser.show(results)